# KerrGeodesics: 56 timelike orbit cases

This Julia notebook gives one parameter set for every catalogue orbit: the six classes Stable (**A1–A2**), Critical (**K1–K11**), Plunge (**B1–B9**), Capture (**C1–C12**), Scatter (**D1–D2**) and Trapped (**N1–N6**), plus the tier members with two-level names (**A-H1**, **A-X1**, **B-X1**, **C-X1**, **D-H1**, …; class letter, then H for a root on the horizon, P(r₊) = 0, or X for extremal Kerr, |a| = 1). By class: Stable 6, Critical 11, Plunge 11, Capture 16, Scatter 6, Trapped 6. Each parameter set is passed to the family constructor kerr_geodesic(a, (E, Lz, Q)); the notebook then selects the catalogued member and evaluates its trajectory.

The README shows all 56 orbits side by side (example/animations/showcase_all.gif); the animated gallery at the end of this notebook shows them in the same style.

In [1]:
# Locate the notebook support file without activating or instantiating an environment.
support_name = "kerr_geodesics_56_case_support.jl"
search_paths = unique([
    joinpath(pwd(), support_name),
    joinpath(pwd(), "example", support_name),
    joinpath(@__DIR__, support_name),
])
support_index = findfirst(isfile, search_paths)
support_index === nothing && error("Could not locate $(support_name). Open this notebook from the package example directory.")
support_path = search_paths[support_index]
isdefined(Main, :KG56_EXAMPLES) || include(support_path)

(julia_version=VERSION, case_count=length(KG56_EXAMPLES))

(julia_version = v"1.12.6", case_count = 56)

## Classification order

`kerr_geodesic` runs the classification steps listed below before it builds any member. It keeps every allowed radial component, so a single input can produce a family with more than one orbit.

In [2]:
kerr_geo_classification_pipeline()

(:canonical_constants, :metric_limit, :energy_regime, :radial_polynomial_degree, :roots_and_multiplicities, :allowed_radial_dynamical_components, :polar_admissibility, :future_direction, :case_assignment, :family_assembly, :optional_selection)

## Parameters for all 56 cases

Each case has one fixed parameter set. ID is the case ID or the two-level tier-member name; Class is Stable, Critical, Plunge, Capture, Scatter or Trapped. The Roots, Allowed interval and Formula columns come from example/data/catalogue_registry.tsv.


In [3]:
kg56_catalog_table()

ID,Class,a,E,Lz,Q,Roots,Allowed interval,Formula
A1,Stable,0.9,0.96412043,2.8359153,4.5444083,R1<rplus<R2<R3<R4,"[R3,R4]",FF01
A2,Stable,0.9,0.9419811,2.5662714,3.7373387,R1<rplus<R2<R3=R4,{R3},FF02_CONSTANT
A-H1,Stable,0.9,0.94,2.9994144,0,R1<rplus=R2<R3<R4,"[R3,R4]",HC_FF01
A-H2,Stable,0.9,0.93311282,2.9774384,0,R1<rplus=R2<R3=R4;Rsecond<0,{R3},HC_CONSTANT
A-X1,Stable,1,0.8,1.6,1,R1=R2=rpm<R3<R4;CH<0;Q<Qsph,"[R3,R4]",EXT_H2_ISLAND
A-X2,Stable,1,0.8,1.6,1.1377778,R1=R2=rpm<R3=R4;Q=Qsph,{R3},EXT_CONSTANT
K1,Critical,0.9,0.87802458,1.923635,2.1482566,R1<rplus<R2=R3=R4,{R2},FF03_CONSTANT
K2,Critical,0.9,0.87802458,1.923635,2.1482566,R1<rplus<R2=R3=R4,"(rplus,R2]",FF03
K3,Critical,0.7,0.91713003,2.2591914,2.8989945,R1<rplus<R2=R3<R4;R=Rprime=0;Rsecond>0,{R2},FF17_CONSTANT
K4,Critical,0.7,0.91713003,2.2591914,2.8989945,R1<rplus<R2=R3<R4,"(R3,R4]",FF17


## Run every case

This cell constructs all 56 families, checks the case IDs the classifier returns, selects the catalogued member, and evaluates finite trajectory points inside r <= 30M. No differential equation is integrated: r(λ) and z(λ) are closed forms, and t, φ, τ (and v, ψ) are Chebyshev primitives of the exact rates.

In [4]:
catalog = kg56_sample_catalog(sample_count=10000, radial_cutoff=30.0)
@assert length(catalog) == 56
@assert all(row -> row.pass, catalog)
kg56_sample_table(catalog)

ID,Status,Case IDs,Retained / requested,Trimmed past / future,min r,max r,Frames / azimuth,Radial residual
A1,PASS,"(:A1, :B1)",10000 / 10000,0 / 0,6.666671,19.999953,t / phi,2.7105054e-20
A2,PASS,"(:A2, :B2)",10000 / 10000,0 / 0,8,8,t / phi,7.6294526e-13
A-H1,PASS,"(:A_H1,)",10000 / 10000,0 / 0,5.3243003,10.421941,t / phi,0
A-H2,PASS,"(:A_H2,)",10000 / 10000,0 / 0,7.0159797,7.0159797,t / phi,2.8664632e-13
A-X1,PASS,"(:A_X1,)",10000 / 10000,0 / 0,1.1591373,1.5801198,t / phi,3.2311743e-27
A-X2,PASS,"(:A_X2,)",10000 / 10000,0 / 0,1.7777778,1.7777778,t / phi,8.8817842e-16
K1,PASS,"(:K1, :K2)",10000 / 10000,0 / 0,2.7942193,2.7942193,t / phi,9.547918e-15
K2,PASS,"(:K1, :K2)",9999 / 10000,1 / 0,1.4358899,2.7178111,v / psi,5.5511151e-17
K3,PASS,"(:K3, :K4, :K5)",10000 / 10000,0 / 0,3.2265813,3.2265813,t / phi,4.8405724e-14
K4,PASS,"(:K3, :K4, :K5)",9998 / 10000,1 / 1,3.2271093,5.9922212,t / phi,0


## Inspect one case

Change selected_case to any ID from the table (a Symbol such as :B1 or :D_H1, or a display name such as "D-H1") and rerun the next three cells.


In [5]:
selected_case = :B1

:B1

In [6]:
selected = kg56_construct(selected_case)
kg56_case_summary(selected_case)

Case,B1 (Plunge)
Input,"(a=0.9, E=0.9641204328952226, Lz=2.8359152778998453, Q=4.544408272395823)"
Case IDs,"(:A1, :B1)"
Selected member,plunge[1]
Member IDs,"(:B1,)"
Root structure,R1<rplus<R2<R3<R4
Allowed interval,"(rplus,R2]"
Formula family,FF01


In [ ]:
kg56_orbit_plot(selected_case; sample_count=10000, radial_cutoff=30.0)

**Figure.** The selected case (its $a, E, L_z, Q$ are printed above the panels). Left: the $x$–$y$ projection; middle: the $x$–$z$ projection, with $x = r\sin\theta\cos\phi$, $y = r\sin\theta\sin\phi$, $z = r\cos\theta$ (units of $M$, spin along $z$), with $\phi$ the azimuth $\varphi$, or $\psi = \varphi + \varphi_H$ for orbits that cross a horizon; the disc is the outer horizon $r_+$. Right: the radius $r$ (vertical axis) against Mino time $\lambda$ (horizontal axis) over the plotted window.

The selected objects remain available for direct inspection:

- selected.family is the complete KerrGeodesicFamily.
- selected.member is the selected member, a `KerrGeoComponent{C}` of its class (Stable, Critical, Plunge, Capture, Scatter, or the future half of a Trapped orbit).
- selected.member.Trajectory contains the coordinate callables (t, r, theta, z, phi, tau, and v, psi where the member reaches the horizon); selected.member.Velocity the Mino-time rates.


## Classify custom constants

Edit the four constants below. The output lists the family's members by display name and by case Symbol. A family may contain several members.


In [8]:
custom_input = (a=0.7, E=1.0, Lz=4.0, Q=3.0)
custom = kg56_custom_classification(
    custom_input.a,
    custom_input.E,
    custom_input.Lz,
    custom_input.Q,
)
(input=custom.input,
 case_names=custom.case_names,
 case_ids=custom.case_ids,
 family_type=typeof(custom.family))

(input = (a = 0.7, E = 1.0, Lz = 4.0, Q = 3.0), case_names = ("B5", "D1"), case_ids = (:B5, :D1), family_type = KerrGeodesicFamily)

In [9]:
# This input contains both a horizon-side plunge (B5) and an infinity-side scatter branch (D1).
custom_plunge = custom.family.Plunge
custom_scatter = custom.family.Scatter
custom_samples = (
    plunge=kg56_sample(custom_plunge; sample_count=10000, radial_cutoff=30.0),
    scatter=kg56_sample(custom_scatter; sample_count=10000, radial_cutoff=30.0),
)
(plunge_points=length(custom_samples.plunge.lambda),
 scatter_points=length(custom_samples.scatter.lambda),
 plunge_interval=custom_plunge.Domain.mino,
 scatter_interval=custom_scatter.Domain.mino)

(plunge_points = 10000, scatter_points = 6820, plunge_interval = (0.0, 0.06296448265701471), scatter_interval = (-0.8532875767867478, 0.8532875767867478))

## Static and animated views of all 56 trajectories

The first cell draws a compact static overview (x–y projections). The final cell animates all 56 orbits in the style of the README animation: each panel has a fixed orthographic camera (elevation and azimuth from example/views.tsv), the event horizon r+ with its glow, the ergosphere wireframe and the spin axis, the whole path in faint colour and a fading trail in the class palette; parts behind the horizon are hidden. Horizon-ending branches are resampled uniformly in advanced time v and plotted with ingoing azimuth psi; the remaining branches are resampled uniformly in coordinate time t and use phi. The particle advances uniformly in a blend of that time and arc length, and the trail covers the last quarter of the time. Orbits that stay close to the horizon are drawn with r − r+ magnified (factor on the panel). On branches that end on a future horizon the last quarter of the Mino-time grid is clustered toward the horizon; next to a repeated-root asymptote, samples within 10⁻⁷ (relative) of the limiting radius are not drawn. The panels are drawn by a short script in the output; where output scripts do not run (an untrusted notebook, a static viewer) the static layer of each panel still shows the orbit.

In [ ]:
kg56_gallery(catalog)

**Figure.** The $x$–$y$ projection of every catalogue orbit, with $x = r\sin\theta\cos\phi$, $y = r\sin\theta\sin\phi$, $z = r\cos\theta$ (units of $M$, spin along $z$), with $\phi$ the azimuth $\varphi$, or $\psi = \varphi + \varphi_H$ for orbits that cross a horizon, cut at $r = 30M$; each tile is labelled with its case and motion, and its constants are in the parameter table above.

In [ ]:
kg56_animated_gallery(catalog)

![All 56 catalogue orbits](animations/showcase_all.gif)

**Figure.** All 56 catalogue orbits animated in 3D, with $x = r\sin\theta\cos\phi$, $y = r\sin\theta\sin\phi$, $z = r\cos\theta$ (units of $M$, spin along $z$), with $\phi$ the azimuth $\varphi$, or $\psi = \varphi + \varphi_H$ for orbits that cross a horizon. Each panel gives the spin $a$ and constants $(E,L_z,Q)$. Particle progress uses an equal blend of normalized time and arc length, with time $t$ or advanced time $v=t+r_*$ for horizon-crossing orbits; it is not uniform in physical time. Where marked, the displayed distance from the horizon is magnified by the panel's factor. The black sphere is the outer horizon and the wireframe the ergosphere.

## Arbitrary precision

The catalogue runs in `Float64`, the default. Any case can be built in `BigFloat` with the keyword `precision = p` (bits). The cell below builds B1 at 256 bits from the catalogue constants and compares it with the `Float64` member: the differences are the `Float64` rounding errors. Cases defined by a repeated root or a root on the horizon (Critical, the H and X tiers) need their constants computed in the target precision, since the `Float64` constants are degenerate only to `Float64` rounding.

In [12]:
# B1 at 256 bits from the catalogue constants (Float64 numbers, converted exactly).
example_b1 = kg56_example(:B1)
fam256 = kerr_geodesic(example_b1.a, (example_b1.E, example_b1.Lz, example_b1.Q);
    precision=256, example_b1.kwargs...)
m256 = fam256.Plunge
m64 = kg56_construct(:B1).member
println("Case: ", m256.CaseId, ", class: ", fam256.BroadClass)
lo, hi = m64.Domain.mino                    # B1 ends on the future horizon at λ = hi
for λ in lo .+ (hi - lo) .* (0:4) ./ 5
    @printf("λ = %+.3f   r = %.20f   |Δr| = %.1e   |Δt| = %.1e\n", λ, m256.Trajectory.r(big(λ)),
        abs(Float64(m256.Trajectory.r(big(λ)) - m64.Trajectory.r(λ))),
        abs(Float64(m256.Trajectory.t(big(λ)) - m64.Trajectory.t(λ))))
end

Case: B1, class: plunge
λ = +0.000   r = 1.44176900939399523362   |Δr| = 1.8e-17   |Δt| = 0.0e+00
λ = +0.011   r = 1.44153357435260403434   |Δr| = 6.2e-17   |Δt| = 5.7e-17
λ = +0.022   r = 1.44082740437169529448   |Δr| = 2.5e-17   |Δt| = 3.4e-16
λ = +0.033   r = 1.43965090499068746181   |Δr| = 8.7e-17   |Δt| = 3.4e-16
λ = +0.043   r = 1.43800475247350159360   |Δr| = 2.5e-17   |Δt| = 8.3e-16
